# Análise qualitativa de casos ruins

## Objetivo

Seleciona casos ruins e gera comparações qualitativas 3D da aorta, dos óstios e da segmentação arterial em mid e high resolution.

In [ ]:
# ruff: noqa: E402
import sys
from pathlib import Path

NOTEBOOK_CWD = Path.cwd().resolve()
for candidate in (
    NOTEBOOK_CWD,
    NOTEBOOK_CWD.parent,
    NOTEBOOK_CWD.parent.parent,
):
    src_dir = candidate / "src"
    if src_dir.exists():
        if str(src_dir) not in sys.path:
            sys.path.insert(0, str(src_dir))
        break

from utils.project.notebook_env import configure_notebook_environment  # noqa: E402

REPO_ROOT = configure_notebook_environment()


In [ ]:
from utils.comparison_utils import prepare_bad_case_qualitative_comparison
from utils.experiments import run_qualitative_pipeline_case
from utils.project.notebook_env import (
    get_bad_cases_export_dir,
    get_cases_analysis_output_dir,
    get_default_split_paths,
    resolve_imagecas_base_path,
)
from utils import (
    load_config_json,
    scale_config_to_resolution,
    visualize_aorta_with_ostia,
    visualize_arteries_comparison,
)


## Configuração

Ajuste nesta seção apenas os parâmetros da análise; o pipeline base não é alterado.

## Carregamento

Carrega ou constrói os dados necessários para as análises seguintes.

In [ ]:
# Configurações
CONFIG_PATH = REPO_ROOT / "config/pipeline_config.json"
DATA_PATH = resolve_imagecas_base_path()
BAD_CASES_EXPORT_DIR = get_bad_cases_export_dir(REPO_ROOT)
SPLIT_PATHS_BY_RESOLUTION = get_default_split_paths(REPO_ROOT)
HTML_OUTPUT_DIR = get_cases_analysis_output_dir(REPO_ROOT)
HTML_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RANDOM_SEED = 42
SUMMARY_SPLIT = "test"
N_SAMPLES_PER_ERROR_INTERSECTION = 2

# Carrega a configuração padrão do pipeline.
CONFIG = load_config_json(str(CONFIG_PATH), {})

comparison_data = prepare_bad_case_qualitative_comparison(
    SPLIT_PATHS_BY_RESOLUTION,
    BAD_CASES_EXPORT_DIR,
    split_name=SUMMARY_SPLIT,
    resolutions=("high", "mid"),
    samples_per_group=N_SAMPLES_PER_ERROR_INTERSECTION,
    random_seed=RANDOM_SEED,
)
RESOLUTIONS = comparison_data["resolutions"]
summaries_by_resolution = comparison_data["summaries_by_resolution"]
bad_cases_by_resolution = comparison_data["bad_cases_by_resolution"]
all_bad_cases = comparison_data["all_bad_cases"]
selected_image_plan = comparison_data["selected_image_plan"]
selected_image_ids = comparison_data["selected_image_ids"]
selected_cases = comparison_data["selected_cases"]

print(f"Total de {len(selected_image_ids)} imagens selecionadas para comparação")
print(f"Total de {len(selected_cases)} execuções planejadas (mid/high por imagem)")
print("\nPlano de amostragem por tipo de erro e interseção:")
display(
    selected_image_plan.sort_values(
        ["target_bad_case_status", "target_intersection_group", "image_id"]
    )
)

print("\nComparação dos status selecionados por resolução:")
display(
    selected_cases.pivot_table(
        index=["image_id", "selection_reason"],
        columns="resolution",
        values="bad_case_status",
        aggfunc="first",
    ).reset_index()
)

print("\nDetalhes dos casos que serão processados:")
display(
    selected_cases[
        [
            "image_id",
            "resolution",
            "bad_case_status",
            "intersection_group",
            "status",
            "ostia_status",
            "left_intersects",
            "right_intersects",
            "dice_artery",
            "selection_reason",
        ]
    ].sort_values(["image_id", "resolution"])
)

## Análise

As subseções abaixo apresentam as métricas, tabelas ou visualizações do objetivo definido.

## Visualização de aorta, óstios e artérias

In [ ]:
pipeline_results = {}

for idx, row in selected_cases.iterrows():
    img_id = int(row['image_id'])
    resolution = row['resolution']
    bad_case_status = row['bad_case_status']

    if resolution == 'high':
        CONFIG["DOWNSCALE_FACTORS"] = [1, 1, 1]
    else:
        CONFIG["DOWNSCALE_FACTORS"] = [2, 2, 1]

    scaled_config = scale_config_to_resolution(CONFIG)
    try:
        result = run_qualitative_pipeline_case(
            img_id,
            scaled_config,
            DATA_PATH,
        )
    except Exception as exc:
        result = {"success": False, "error": str(exc)}
    pipeline_results[(img_id, resolution)] = result

    if not isinstance(result, dict) or result.get('success', True) is False:
        print(f"IMG {img_id} ({resolution}) - falhou: {result.get('error', 'resultado inválido') if isinstance(result, dict) else 'resultado inválido'}")
        continue

    error_type_clean = bad_case_status.lower().replace(' ', '_').replace('/', '_')
    ostia_left = result['ostia_left']
    ostia_right = result['ostia_right']
    aorta_mask = result['aorta_mask']
    label_artery = result['label_artery']
    spacing = result['scaled_spacing']

    try:
        ostia_filename = f"img_{img_id}_{resolution}_{error_type_clean}_aorta_ostios.html"
        ostia_path = HTML_OUTPUT_DIR / ostia_filename
        visualize_aorta_with_ostia(
            aorta_mask,
            ostia_left,
            ostia_right,
            spacing=spacing,
            label_mask=label_artery,
            use_physical_coords=True,
            save_html_path=ostia_path,
            display_plot=False,
        )
    except Exception as e:
        print(f"IMG {img_id} ({resolution}) - erro ao gerar HTML de óstios: {e}")

    try:
        artery_filename = f"img_{img_id}_{resolution}_{error_type_clean}_arteries_comparison.html"
        artery_path = HTML_OUTPUT_DIR / artery_filename
        artery_mask = result['artery_results']['artery_mask']
        visualize_arteries_comparison(
            label_mask=label_artery,
            predicted_mask=artery_mask,
            spacing=spacing,
            save_html_path=artery_path,
            display_plot=False,
        )
    except Exception as e:
        print(f"IMG {img_id} ({resolution}) - erro ao gerar comparação de artérias: {e}")

print(f"Visualizações concluídas para {len(selected_cases)} casos")


## Conclusão

As visualizações 3D permitem confirmar visualmente a origem dos erros apontados pela análise quantitativa e comparar o comportamento entre resoluções disponíveis.